# Project Management Analysis

This notebook presents an exploratory data analysis (EDA) and predictive modeling on a synthetic project management dataset. The goal is to assess factors that influence project success and develop models to predict the likelihood of a project being successful based on various features.



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# display plots inline
%matplotlib inline


In [ ]:
# Load the dataset
file_path = 'project_management_data.csv'
df = pd.read_csv(file_path)

# Display the first few rows
df.head()


In [ ]:
# Summary statistics
df.describe(include='all')


In [ ]:
# Visualize distributions of numeric features
numeric_cols = ['team_size','project_budget','actual_budget','planned_duration','actual_duration','issues_reported','satisfaction_score','manager_experience_years']

plt.figure(figsize=(12, 8))
for i, col in enumerate(numeric_cols):
    plt.subplot(3, 3, i+1)
    sns.histplot(df[col], kde=True)
    plt.title(col)
plt.tight_layout()
plt.show()

# Correlation heatmap for numeric features
plt.figure(figsize=(8,6))
corr = df[numeric_cols + ['success']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()


In [ ]:
# Categorical features and success rate
categorical_cols = ['complexity','industry']

for col in categorical_cols:
    plt.figure(figsize=(6,4))
    sns.barplot(x=col, y='success', data=df, ci=None)
    plt.title(f'Success rate by {col}')
    plt.ylabel('Average Success Rate')
    plt.xlabel(col.capitalize())
    plt.show()


In [ ]:
# Prepare data for modeling
target = 'success'
X = df.drop(columns=[target, 'project_id'])
y = df[target]

# Identify categorical and numerical columns
cat_cols = ['complexity','industry']
num_cols = [col for col in X.columns if col not in cat_cols]

# Preprocess data: one-hot encode categorical variables
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ], remainder='passthrough'
)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Logistic Regression model
log_reg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

log_reg_pipeline.fit(X_train, y_train)
y_pred_lr = log_reg_pipeline.predict(X_test)

print("Logistic Regression Model Performance:")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

# Random Forest model
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=200, random_state=42))
])

rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)

print("Random Forest Model Performance:")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

# Display confusion matrix for the better performing model
best_model_name = 'Random Forest' if accuracy_score(y_test, y_pred_rf) > accuracy_score(y_test, y_pred_lr) else 'Logistic Regression'
y_pred_best = y_pred_rf if best_model_name == 'Random Forest' else y_pred_lr

conf_mat = confusion_matrix(y_test, y_pred_best)
sns.heatmap(conf_mat, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix ({best_model_name})')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


## Conclusion

This synthetic project management analysis demonstrates how various factors—such as team size, budget adherence, duration, and qualitative aspects like complexity and industry—can influence project success. Through exploratory data visualization, we identified patterns and relationships among the features. We then trained logistic regression and random forest models to predict project success. The random forest model achieved higher accuracy in this scenario, indicating its effectiveness in capturing nonlinear relationships in the data.

You can experiment further by tuning model hyperparameters, exploring additional algorithms, or augmenting the dataset with more features to simulate real-world project scenarios.
